In [11]:
import json
import pandas as pd
import altair as alt
from scipy.stats import chi2_contingency
import statsmodels.formula.api as smf
alt.data_transformers.disable_max_rows()
from IPython.display import Markdown, display
import numpy as np


In [12]:
summarydf = pd.read_csv("./../output/numeric_summary_by_variant.csv")
summarydf


with open("./../results_fixed.json") as f:
    data = json.load(f)

df = pd.json_normalize(data)
print("Number of rows:", len(df))
df.head()

df.columns.tolist()

rename_map = {
    'category_counts.attack': 'category_attack',
    'category_counts.defense': 'category_defense',
    'category_counts.terrorism': 'category_terrorism',
    'category_counts.retaliation': 'category_retaliation',
    'category_counts.humanitarian': 'category_humanitarian',
    'category_counts.diplomacy': 'category_diplomacy',

    'actor_sentiment.palestine.mean_score': 'sent_pal_mean',
    'actor_sentiment.palestine.label': 'sent_pal_label',
    'actor_sentiment.palestine.adjectives': 'sent_pal_adj',
    'actor_sentiment.israel.mean_score': 'sent_isr_mean',
    'actor_sentiment.israel.label': 'sent_isr_label',
    'actor_sentiment.israel.adjectives': 'sent_isr_adj',

    'actor_adjectives.palestine': 'adj_pal',
    'actor_adjectives.israel': 'adj_isr',

    'narrative_metrics.palestine_mentions': 'mentions_pal',
    'narrative_metrics.palestine_adj_count': 'adj_pal_count',
    'narrative_metrics.palestine_mentions_ratio': 'mentions_pal_ratio',
    'narrative_metrics.palestine_adj_density': 'adj_pal_density',

    'narrative_metrics.israel_mentions': 'mentions_isr',
    'narrative_metrics.israel_adj_count': 'adj_isr_count',
    'narrative_metrics.israel_mentions_ratio': 'mentions_isr_ratio',
    'narrative_metrics.israel_adj_density': 'adj_isr_density',

    'narrative_metrics.mentions_asymmetry': 'mentions_asymmetry',
    'narrative_metrics.adj_asymmetry': 'adj_asymmetry',

    'lexical_metrics.ttr': 'ttr',
    'lexical_metrics.mtld': 'mtld',
    'lexical_metrics.mean_word_length': 'mean_word_length',
    'lexical_metrics.sentence_count': 'sentence_count',
    'lexical_metrics.char_count': 'char_count',

    'assertiveness.absolutist_count': 'abs_count',
    'assertiveness.hedge_count': 'hedge_count'
}

df_alt = df.rename(columns=rename_map)
df_alt.columns


Number of rows: 100


Index(['variant', 'prompt_id', 'prompt', 'response', 'timestamp',
       'original_text', 'sentiment_score', 'sentiment_label', 'token_count',
       'mean_token_score', 'token_level_scores', 'pos_word_hits',
       'neg_word_hits', 'neu_word_hits', 'causal_attributions', 'actor_events',
       'composite_label', 'category_attack', 'category_defense',
       'category_terrorism', 'category_retaliation', 'category_humanitarian',
       'category_diplomacy', 'sent_pal_mean', 'sent_pal_label', 'sent_pal_adj',
       'sent_isr_mean', 'sent_isr_label', 'sent_isr_adj', 'adj_pal', 'adj_isr',
       'mentions_pal', 'adj_pal_count', 'mentions_pal_ratio',
       'adj_pal_density', 'mentions_isr', 'adj_isr_count',
       'mentions_isr_ratio', 'adj_isr_density', 'mentions_asymmetry',
       'adj_asymmetry', 'ttr', 'mtld', 'mean_word_length', 'sentence_count',
       'char_count', 'abs_count', 'hedge_count'],
      dtype='object')

In [13]:
df = df[df["variant"].notna()].copy()
#sanity
df["variant"].value_counts()


variant
base_model         25
pro_israeli        25
pro_palestinian    25
neutral            25
Name: count, dtype: int64

In [14]:
chart_sent_dist = (
    alt.Chart(df_alt)
    .mark_boxplot()
    .encode(
        x=alt.X("variant:N", title="Variant"),
        y=alt.Y("sentiment_score:Q", title="Sentiment score (RoBERTa)"),
        color="variant:N"
    )
    .properties(
        title="Distribution of Sentiment Scores by Variant",
        width=400,
        height=300
    )
)

chart_sent_dist

alt.Chart(...)

In [15]:
sent_label_counts = (
    df_alt.groupby(["variant", "sentiment_label"])
    .size()
    .reset_index(name="count")
)

sent_label_counts

chart_sent_labels = (
    alt.Chart(sent_label_counts)
    .mark_bar()
    .encode(
        x=alt.X("variant:N", title="Variant"),
        y=alt.Y("count:Q", title="Count"),
        color=alt.Color("sentiment_label:N", title="Sentiment label"),
        column=alt.Column("sentiment_label:N", title=None)
    )
    .properties(
        title="Sentiment Label Distribution by Variant",
        width=150,
        height=300
    )
)

chart_sent_labels



alt.Chart(...)

In [16]:
asym_col = "mentions_asymmetry"

asym_agg = (
    df_alt.groupby("variant")[asym_col]
          .mean()
          .reset_index()
)

display(asym_agg)

chart_mentions_asym = (
    alt.Chart(asym_agg)
    .mark_bar()
    .encode(
        x=alt.X("variant:N", title="Variant"),
        y=alt.Y(f"{asym_col}:Q",
                title="Mean mentions asymmetry (Palestine - Israel, normed)"),
        color="variant:N",
        tooltip=["variant", asym_col]
    )
    .properties(
        title="Mean Narrative Mentions Asymmetry by Variant",
        width=400,
        height=300
    )
)

chart_mentions_asym


,variant,mentions_asymmetry
0,base_model,-0.258364
1,neutral,-0.349039
2,pro_israeli,-0.259701
3,pro_palestinian,-0.300457


alt.Chart(...)

In [17]:
chart_mentions_asym = (
    alt.Chart(df_alt)
    .mark_boxplot()
    .encode(
        x=alt.X("variant:N", title="Variant", axis=alt.Axis(labelAngle=0)),
        y=alt.Y(f"{asym_col}:Q",
                title="Mentions Asymmetry (Palestine − Israel, normalized)"),
        color=alt.Color("variant:N", legend=None),
        tooltip=["variant", asym_col]
    )
    .properties(
        title="Narrative Mentions Asymmetry",
        width=400,
        height=280
    )
)

chart_mentions_asym

alt.Chart(...)

In [18]:
terror_col = "category_terrorism"
agg_terror = (
    df_alt.groupby("variant")[terror_col]
    .sum()
    .reset_index()
)

agg_terror

chart_terror = (
    alt.Chart(agg_terror)
    .mark_bar()
    .encode(
        x=alt.X("variant:N", title="Variant"),
        y=alt.Y(f"{terror_col}:Q", title="Total 'terrorism' keyword hits"),
        color="variant:N"
    )
    .properties(
        title="Total 'Terrorism' Keyword Usage by Variant",
        width=400,
        height=300
    )
)

chart_terror



alt.Chart(...)

In [19]:
mtld_col = "mtld"

chart_mtld = (
    alt.Chart(df_alt)
    .mark_boxplot()
    .encode(
        x=alt.X("variant:N", title="Variant"),
        y=alt.Y(f"{mtld_col}:Q", title="MTLD (lexical diversity)"),
        color="variant:N"
    )
    .properties(
        title="Lexical Diversity (MTLD) by Variant",
        width=400,
        height=300
    )
)

chart_mtld


alt.Chart(...)

In [20]:
mtld_sent_scatter = (
    alt.Chart(df_alt)
    .mark_circle(opacity=0.4, size=70)
    .encode(
        x=alt.X("mtld:Q", title="MTLD (lexical diversity)"),
        y=alt.Y("sentiment_score:Q", title="Sentiment score (RoBERTa)"),
        color=alt.Color("variant:N", title="Variant"),
        tooltip=["variant", "mtld", "sentiment_score", "prompt_id"]
    )
    .properties(
        title="MTLD vs Sentiment Score by Variant",
        width=400,
        height=300
    )
)

mtld_sent_reg = (
    alt.Chart(df_alt)
    .transform_regression(
        "mtld", "sentiment_score", groupby=["variant"]
    )
    .mark_line(size=3)
    .encode(
        x="mtld:Q",
        y="sentiment_score:Q",
        color=alt.Color("variant:N", title="Variant")
    )
)

(mtld_sent_scatter + mtld_sent_reg).resolve_scale(color="independent")


alt.LayerChart(...)

In [21]:
print("Overall correlation (MTLD vs sentiment_score):")
display(df_alt[["mtld", "sentiment_score"]].corr())

print("\nPer-variant correlations:")
corr_rows = []
for v, sub in df_alt.groupby("variant"):
    corr = sub[["mtld", "sentiment_score"]].corr().iloc[0, 1]
    corr_rows.append({"variant": v, "corr_mtld_sent": corr})

corr_df = pd.DataFrame(corr_rows)
display(corr_df)


Overall correlation (MTLD vs sentiment_score):


,mtld,sentiment_score
mtld,1.000000,-0.120747
sentiment_score,-0.120747,1.000000



Per-variant correlations:


,variant,corr_mtld_sent
0,base_model,-0.235029
1,neutral,-0.489397
2,pro_israeli,0.005597
3,pro_palestinian,0.110287


In [22]:
mtld_asym_scatter = (
    alt.Chart(df_alt)
    .mark_circle(size=70, opacity=0.45)
    .encode(
        x=alt.X("mtld:Q", title="MTLD (lexical diversity)"),
        y=alt.Y("mentions_asymmetry:Q",
                title="Mentions asymmetry (Palestine − Israel)"),
        color=alt.Color("variant:N", title="Variant"),
        tooltip=["variant", "mtld", "mentions_asymmetry", "prompt_id"]
    )
    .properties(
        title="Linguistic Complexity vs Narrative Asymmetry",
        width=400,
        height=300
    )
)

# Optional: regression lines per variant
mtld_asym_reg = (
    alt.Chart(df_alt)
    .transform_regression(
        "mtld", "mentions_asymmetry", groupby=["variant"]
    )
    .mark_line(size=3)
    .encode(
        x="mtld:Q",
        y="mentions_asymmetry:Q",
        color=alt.Color("variant:N", title="Variant")
    )
)

# Horizontal reference line at 0 (balanced mentions)
zero_rule = alt.Chart(pd.DataFrame({"y": [0]})).mark_rule(strokeDash=[4,4]).encode(
    y="y:Q"
)

mtld_asym_scatter + mtld_asym_reg + zero_rule


alt.LayerChart(...)

In [23]:
print("Overall correlation (MTLD vs mentions_asymmetry):")
display(df_alt[["mtld", "mentions_asymmetry"]].corr())

print("\nPer-variant correlations (complexity vs narrative bias):")
rows = []
for v, sub in df_alt.groupby("variant"):
    corr = sub[["mtld", "mentions_asymmetry"]].corr().iloc[0, 1]
    rows.append({"variant": v, "corr_mtld_asym": corr})

complexity_bias_corr = pd.DataFrame(rows)
display(complexity_bias_corr)

Overall correlation (MTLD vs mentions_asymmetry):


,mtld,mentions_asymmetry
mtld,1.000000,0.019121
mentions_asymmetry,0.019121,1.000000



Per-variant correlations (complexity vs narrative bias):


,variant,corr_mtld_asym
0,base_model,-0.037050
1,neutral,-0.142639
2,pro_israeli,0.189841
3,pro_palestinian,0.060635


In [24]:
def safe_len(x):
    if isinstance(x, (list, tuple)):
        return len(x)
    if pd.isna(x):
        return 0
    return 0

df_alt["causal_count"] = df_alt["causal_attributions"].apply(safe_len)

df_alt[["variant", "sentiment_score", "sentiment_label", "causal_count"]].head()

causal_sent_scatter = (
    alt.Chart(df_alt)
    .mark_circle(size=70, opacity=0.7)
    .encode(
        x=alt.X("causal_count:Q", title="Number of causal attributions detected"),
        y=alt.Y("sentiment_score:Q", title="Sentiment score (RoBERTa)"),
        color=alt.Color(
            "sentiment_label:N",
            title="Sentiment label",
            scale=alt.Scale(scheme="redblue")
        ),
        tooltip=["variant", "sentiment_label", "causal_count",
                 "sentiment_score", "prompt_id"]
    )
    .properties(
        title="Sentiment vs Causal Attribution Density",
        width=450,
        height=320
    )
)

df_alt["has_causal"] = (df_alt["causal_count"] > 0).astype(int)

causal_box = (
    alt.Chart(df_alt)
    .mark_boxplot()
    .encode(
        x=alt.X("has_causal:N", title="Has causal attribution (0 = no, 1 = yes)"),
        y=alt.Y("sentiment_score:Q", title="Sentiment score (RoBERTa)"),
        color=alt.Color("has_causal:N", legend=None)
    )
    .properties(
        title="Sentiment Distribution With vs Without Causal Attributions",
        width=380,
        height=300
    )
)

causal_box



alt.Chart(...)

In [25]:
group_means = (
    df_alt.groupby("has_causal")["sentiment_score"]
          .mean()
          .rename({0:"no_causal", 1:"has_causal"})
)
print(group_means)

diff = group_means[1] - group_means[0]
print(f"\nMean sentiment difference (has_causal - no_causal): {diff:.3f}")

import statsmodels.formula.api as smf

print("======= OLS: sentiment_score ~ causal_count =======")

model_simple = smf.ols("sentiment_score ~ causal_count", data=df_alt).fit()
print(model_simple.summary())

print("\n======= OLS: sentiment_score ~ causal_count + C(variant) =======")

model_variant = smf.ols("sentiment_score ~ causal_count + C(variant)", data=df_alt).fit()
print(model_variant.summary())

print("\n======= OLS: sentiment_score ~ causal_count + mtld + C(variant) =======")

model_full = smf.ols("sentiment_score ~ causal_count + mtld + C(variant)", data=df_alt).fit()
print(model_full.summary())



has_causal
no_causal    -4.539640
has_causal   -6.048666
Name: sentiment_score, dtype: float64

Mean sentiment difference (has_causal - no_causal): -1.509
======= OLS: sentiment_score ~ causal_count =======
                            OLS Regression Results                            
Dep. Variable:        sentiment_score   R-squared:                       0.040
Model:                            OLS   Adj. R-squared:                  0.030
Method:                 Least Squares   F-statistic:                     4.097
Date:                Fri, 05 Dec 2025   Prob (F-statistic):             0.0457
Time:                        22:54:14   Log-Likelihood:                -267.36
No. Observations:                 100   AIC:                             538.7
Df Residuals:                      98   BIC:                             543.9
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
   

/var/folders/ys/2yh_kw0938l96bcd357qq56w0000gn/T/ipykernel_87776/3414934912.py:8: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  diff = group_means[1] - group_means[0]


In [26]:
print("======= Chi-square: sentiment_label ~ variant =======")

ct = pd.crosstab(df_alt["variant"], df_alt["sentiment_label"])
print("Contingency table:")
display(ct)

chi2, p, dof, expected = chi2_contingency(ct)
print(f"\nChi-square = {chi2:.3f}, dof = {dof}, p-value = {p:.5f}")

if p < 0.05:
    print("→ Significant association between variant and sentiment label (α=0.05).")
else:
    print("→ No statistically significant association (α=0.05).")


======= Chi-square: sentiment_label ~ variant =======
Contingency table:


sentiment_label,negative,neutral,positive
variant,,,
base_model,18,7,0
neutral,18,7,0
pro_israeli,21,1,3
pro_palestinian,20,4,1



Chi-square = 11.561, dof = 6, p-value = 0.07251
→ No statistically significant association (α=0.05).


In [27]:
print("======= OLS: sentiment_score ~ C(variant) =======")

model_sent = smf.ols("sentiment_score ~ C(variant)", data=df_alt).fit()
print(model_sent.summary())


======= OLS: sentiment_score ~ C(variant) =======
                            OLS Regression Results                            
Dep. Variable:        sentiment_score   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                 -0.024
Method:                 Least Squares   F-statistic:                    0.2260
Date:                Fri, 05 Dec 2025   Prob (F-statistic):              0.878
Time:                        22:54:14   Log-Likelihood:                -269.06
No. Observations:                 100   AIC:                             546.1
Df Residuals:                      96   BIC:                             556.5
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------

In [28]:
print("======= OLS: category_counts.terrorism ~ C(variant) =======")

model_terror = smf.ols("category_terrorism ~ C(variant)", data=df_alt).fit()
print(model_terror.summary())

import statsmodels.api as sm
import statsmodels.formula.api as smf

# Poisson model
poisson_model = smf.glm(
    formula="category_terrorism ~ C(variant)",
    data=df_alt,
    family=sm.families.Poisson()
).fit()

print("\n===== Poisson Regression: Terrorism Count ~ Variant =====")
print(poisson_model.summary())

import statsmodels.api as sm
import statsmodels.formula.api as smf

# Negative Binomial model
neg_bin_model = smf.glm(
    formula="category_terrorism ~ C(variant)",
    data=df_alt,
    family=sm.families.NegativeBinomial()
).fit()

print("\n===== Negative Binomial Regression: Terrorism Count ~ Variant =====")
print(neg_bin_model.summary())


======= OLS: category_counts.terrorism ~ C(variant) =======
                            OLS Regression Results                            
Dep. Variable:     category_terrorism   R-squared:                       0.023
Model:                            OLS   Adj. R-squared:                 -0.008
Method:                 Least Squares   F-statistic:                    0.7524
Date:                Fri, 05 Dec 2025   Prob (F-statistic):              0.524
Time:                        22:54:14   Log-Likelihood:                -146.73
No. Observations:                 100   AIC:                             301.5
Df Residuals:                      96   BIC:                             311.9
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------

/Users/kadenhyatt/.pyenv/versions/3.9.18/lib/python3.9/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


In [29]:
print("======= OLS: narrative_metrics.mentions_asymmetry ~ C(variant) =======")

model_asym = smf.ols("mentions_asymmetry ~ C(variant)", data=df_alt).fit()
print(model_asym.summary())


======= OLS: narrative_metrics.mentions_asymmetry ~ C(variant) =======
                            OLS Regression Results                            
Dep. Variable:     mentions_asymmetry   R-squared:                       0.010
Model:                            OLS   Adj. R-squared:                 -0.021
Method:                 Least Squares   F-statistic:                    0.3290
Date:                Fri, 05 Dec 2025   Prob (F-statistic):              0.804
Time:                        22:54:21   Log-Likelihood:                -41.301
No. Observations:                 100   AIC:                             90.60
Df Residuals:                      96   BIC:                             101.0
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------